### We'll evaluate all 973 golden test questions against both persisted Chroma indexes.

Our objective is to answer one important question:

Does structure-based chunking retrieve government scheme information more effectively than recursive character chunking?

We'll keep the embedding model, query preprocessing, similarity configuration and Top-K identical.
Evaluation design
Configuration	Value
Test questions	973
EXP_001	Recursive Character
EXP_002_v2	Structure-Based
Embedding model	all-MiniLM-L6-v2
Maximum sequence length	256
Embedding batch size	16
Retrieval Top-K	20
Evaluation K	5, 10, 20

We'll calculate chunk-level precision and scheme-level Recall, Precision, MRR and NDCG. This is important because your current ground truth identifies relevant schemes, not exact relevant chunks.

In [1]:

# ============================================================
# EXP_003: DENSE RETRIEVAL EVALUATION
# EXP_001 vs EXP_002_v2
# ============================================================

import json
import math
import pandas as pd
import numpy as np
import chromadb
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)

QUERY_BATCH_SIZE = 32
EMBEDDING_BATCH_SIZE = 16

# Load the same embedding model and configuration
query_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

query_model.max_seq_length = 256

# Connect to existing Chroma indexes
client = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001"
)

collection_001 = client.get_collection("exp_001_recursive")

client_002 = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_002_v2"
)

collection_002 = client_002.get_collection("exp_002_structure_v2")

# Select one golden test question
golden_test_path = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits\\test.json")
with open(golden_test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

# Keep query embedding configuration identical to indexing
query_model.max_seq_length = 256

experiments = {
    "EXP_001": collection_001,
    "EXP_002_v2": collection_002
}


# ------------------------------------------------------------
# 2. PREPARE GOLDEN TEST QUESTIONS
# ------------------------------------------------------------

valid_tests = [
    item for item in test_data
    if item.get("question")
    and item.get("relevant_scheme_ids")
]

questions = [item["question"] for item in valid_tests]

print("Total test questions:", len(questions))
print("Skipped questions:", len(test_data) - len(questions))


# ------------------------------------------------------------
# 3. GENERATE QUERY EMBEDDINGS ONCE
# ------------------------------------------------------------

query_embeddings = query_model.encode(
    questions,
    batch_size=EMBEDDING_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Query embeddings shape:", query_embeddings.shape)


# ------------------------------------------------------------
# 4. METRIC FUNCTIONS
# ------------------------------------------------------------

def calculate_recall(retrieved_ids, relevant_ids):

    if not relevant_ids:
        return 0.0

    hits = len(set(retrieved_ids) & relevant_ids)

    return hits / len(relevant_ids)


def calculate_precision(retrieved_ids, relevant_ids, k):

    hits = sum(
        1 for sid in retrieved_ids[:k]
        if sid in relevant_ids
    )

    return hits / k


def calculate_mrr(retrieved_ids, relevant_ids, k):

    for rank, sid in enumerate(retrieved_ids[:k], 1):

        if sid in relevant_ids:
            return 1.0 / rank

    return 0.0


def calculate_ndcg(retrieved_ids, relevant_ids, k):

    dcg = 0.0

    for rank, sid in enumerate(retrieved_ids[:k], 1):

        relevance = 1 if sid in relevant_ids else 0

        dcg += relevance / math.log2(rank + 1)

    ideal_relevant = min(len(relevant_ids), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    return dcg / idcg if idcg else 0.0


# ------------------------------------------------------------
# 5. RETRIEVAL + EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_collection(collection, experiment_name):

    results_per_question = []

    total_questions = len(valid_tests)

    print(f"\nEvaluating {experiment_name}")

    for start in tqdm(
        range(0, total_questions, QUERY_BATCH_SIZE),
        desc=experiment_name
    ):

        end = min(start + QUERY_BATCH_SIZE, total_questions)

        batch_embeddings = query_embeddings[start:end]

        # Retrieve top 20 chunks for each query
        retrieved = collection.query(
            query_embeddings=batch_embeddings.tolist(),
            n_results=MAX_K,
            include=["metadatas", "distances"]
        )

        for local_idx, result_idx in enumerate(range(start, end)):

            test = valid_tests[result_idx]

            relevant_ids = set(
                test["relevant_scheme_ids"]
            )

            metadata_list = retrieved["metadatas"][local_idx]

            # Extract scheme IDs in ranking order
            chunk_scheme_ids = [
                metadata.get("scheme_id")
                for metadata in metadata_list
                if metadata and metadata.get("scheme_id")
            ]

            # Remove duplicate schemes while preserving rank
            unique_scheme_ids = list(
                dict.fromkeys(chunk_scheme_ids)
            )

            row = {
                "experiment": experiment_name,
                "test_id": test["test_id"],
                "question": test["question"],
                "scheme_id": test["scheme_id"],
                "relevant_scheme_ids": list(relevant_ids)
            }

            for k in K_VALUES:

                # Chunk-level precision
                row[f"chunk_precision@{k}"] = (
                    calculate_precision(
                        chunk_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                # Scheme-level metrics using unique schemes
                row[f"scheme_recall@{k}"] = (
                    calculate_recall(
                        unique_scheme_ids[:k],
                        relevant_ids
                    )
                )

                row[f"scheme_precision@{k}"] = (
                    calculate_precision(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                row[f"scheme_mrr@{k}"] = (
                    calculate_mrr(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                row[f"scheme_ndcg@{k}"] = (
                    calculate_ndcg(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

            results_per_question.append(row)

    return pd.DataFrame(results_per_question)


# ------------------------------------------------------------
# 6. RUN BOTH EXPERIMENTS
# ------------------------------------------------------------

df_001_results = evaluate_collection(
    collection_001,
    "EXP_001"
)

df_002_results = evaluate_collection(
    collection_002,
    "EXP_002_v2"
)


# ------------------------------------------------------------
# 7. COMBINE RESULTS
# ------------------------------------------------------------

all_results = pd.concat(
    [df_001_results, df_002_results],
    ignore_index=True
)


# ------------------------------------------------------------
# 8. GENERATE SUMMARY METRICS
# ------------------------------------------------------------

metric_columns = [
    column
    for column in all_results.columns
    if any(
        column.startswith(prefix)
        for prefix in [
            "chunk_precision",
            "scheme_recall",
            "scheme_precision",
            "scheme_mrr",
            "scheme_ndcg"
        ]
    )
]

summary = (
    all_results
    .groupby("experiment")[metric_columns]
    .mean()
    .reset_index()
)


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

output_dir = Path(
    "experiments/results/EXP_003_dense_retrieval"
)

output_dir.mkdir(parents=True, exist_ok=True)

all_results.to_csv(
    output_dir / "per_question_results.csv",
    index=False
)

summary.to_csv(
    output_dir / "summary.csv",
    index=False
)


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DENSE RETRIEVAL EVALUATION COMPLETED")
print("=" * 65)

print("Evaluated questions:", len(valid_tests))

display(
    summary.round(4)
)

print("\nResults saved to:", output_dir)


g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2377.30it/s]


Total test questions: 973
Skipped questions: 0


Batches: 100%|██████████| 61/61 [00:25<00:00,  2.35it/s]


Query embeddings shape: (973, 384)

Evaluating EXP_001


EXP_001: 100%|██████████| 31/31 [00:11<00:00,  2.59it/s]



Evaluating EXP_002_v2


EXP_002_v2: 100%|██████████| 31/31 [00:10<00:00,  2.89it/s]



DENSE RETRIEVAL EVALUATION COMPLETED
Evaluated questions: 973


,experiment,chunk_precision@5,scheme_recall@5,scheme_precision@5,scheme_mrr@5,scheme_ndcg@5,chunk_precision@10,scheme_recall@10,scheme_precision@10,scheme_mrr@10,scheme_ndcg@10,chunk_precision@20,scheme_recall@20,scheme_precision@20,scheme_mrr@20,scheme_ndcg@20
0,EXP_001,0.2234,0.6886,0.1377,0.5933,0.6171,0.1496,0.7472,0.0747,0.6011,0.6360,0.0960,0.7965,0.0398,0.6047,0.6487
1,EXP_002_v2,0.2179,0.6917,0.1383,0.5986,0.6219,0.1403,0.7523,0.0752,0.6067,0.6415,0.0863,0.8068,0.0403,0.6107,0.6555



Results saved to: experiments\results\EXP_003_dense_retrieval


In [1]:
print(test_data[0])

NameError: name 'test_data' is not defined